# BOT-2 Shape Scoring · Five-Fold Child-Grouped Cross-Validation

## 1. Setup

In [ ]:
%pip install -q -U timm "transformers>=4.56" safetensors

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, importlib
import pandas as pd

DRIVE_DIR = "/content/drive/MyDrive/Evaluation by prof. nam's research group"
FINAL_DIR = os.path.join(DRIVE_DIR, 'final')
ZIP_PATH = os.path.join(DRIVE_DIR, 'Shapes.zip')
N_WORKERS, WORKER = 1, 0          # parallel tabs: N_WORKERS = 3 in every tab and WORKER = 0, 1, 2
SAVE_WEIGHTS = 'ensemble'         # 'ensemble' (about 4 GB), 'all' (about 9 GB) or 'none'
SHUTDOWN_AT_END = True

sys.path.insert(0, FINAL_DIR)
import bot2score
from bot2score import utils, data, models, training, evaluation, baselines, figures, verify
for module in (utils, data, models, training, evaluation, baselines, figures, verify):
    importlib.reload(module)
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.precision', 3)

P = utils.make_paths(os.path.join(FINAL_DIR, 'results'))
log = utils.Log(os.path.join(P.logs, f'run_log_worker{WORKER}.txt'))
if SHUTDOWN_AT_END:
    utils.release_runtime_on_error(log)
env = utils.environment()
utils.save_json(os.path.join(P.logs, f'environment_worker{WORKER}.json'),
                {**env, 'bot2score': bot2score.__version__,
                 'code_sha256': utils.code_hashes(os.path.dirname(bot2score.__file__))})
log(f"bot2score {bot2score.__version__} | {env.get('gpu', 'no GPU')} | torch {env['torch']} | "
    f"timm {env['timm']} | transformers {env['transformers']}")

## 2. Data

In [ ]:
D = data.prepare(ZIP_PATH, P, log=log)
data.score_table(D)

In [ ]:
data.fold_table(D)

## 3. Networks

In [ ]:
models.table()

## 4. Training

In [ ]:
status = training.run(D, P, worker=WORKER, n_workers=N_WORKERS, save=SAVE_WEIGHTS, log=log)
if WORKER != 0 and SHUTDOWN_AT_END:
    utils.release_runtime(log)
status

## 5. Feature-based scorers

In [ ]:
baselines.run(D, P, log=log)

## 6. Out-of-fold predictions

In [ ]:
R = evaluation.collect(D, P, log=log)

## 7. Accuracy and agreement

In [ ]:
S = evaluation.summary(D, R, P)
S

In [ ]:
T = evaluation.item_accuracy(D, R, P)
T

In [ ]:
F = evaluation.fold_accuracy(D, R, P)
F

In [ ]:
M = evaluation.item_metrics(D, R, P)
evaluation.confusion(D, R, P)
M[M.Scorer == models.PROPOSED_LABEL].set_index("Item")

## 8. Paired comparisons

In [ ]:
C = evaluation.compare(D, R, P)
C

## 9. Drawings below full marks

In [ ]:
curves = evaluation.below_full_curves(D, R, P)
S[['Scorer', 'Precision below full', 'Recall below full', 'F1 below full', 'AP below full']]

## 10. Embeddings

In [ ]:
E = evaluation.embeddings(D, R, P, log=log)
E['separation']

## 11. Figures

In [ ]:
figures.setup()

In [ ]:
figures.accuracy(S, F, P, log=log);

In [ ]:
figures.comparisons(C, P, log=log);

In [ ]:
figures.precision_recall(D, R, S, curves, P, log=log);

In [ ]:
figures.item_accuracy(T, S, P, log=log);

In [ ]:
figures.confusion(D, R, P=P, log=log);

In [ ]:
figures.embeddings(E, P, log=log);

## 12. Verification of the saved weights

In [ ]:
verify.check_weights(D, P, log=log)

In [ ]:
verify.manifest(P, log=log).tail()

## 13. Release the runtime

In [ ]:
log('Finished. All results are in', P.root)
if SHUTDOWN_AT_END:
    utils.release_runtime(log)